# Ocean3: MOM6 prescribed-load geometry evolution

Analyse the most recent archive of the Ocean3 shelf-mass wetting/drying experiment. The notebook discovers the latest state from NetCDF time coordinates and plots prescribed geometry, vanishing columns, melt, circulation, tracers, conservation, and stability.

The experiment uses a static horizontal wettable envelope with full-cell vertical vanishing-column wetting/drying. It is not a horizontal fractional-cell grounding-line scheme.

In [ ]:
from pathlib import Path
import json, re, warnings
import netCDF4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import plotly.graph_objects as go
except ImportError:
    go = None
    warnings.warn("plotly unavailable; the optional 3-D plot will be skipped")

CASE_NAME = "ocean3-shelf-mass-wetdry-year"
CONTROL = Path("/scratch/au88/jr5971/mom6-isomip-generated-controls") / CASE_NAME
LAB = Path("/scratch/au88/jr5971/mom6-isomip-generated") / CASE_NAME
ARCHIVE = LAB / "archive" / CASE_NAME

SNAPSHOT_YEARS = (0.0, 10.0, 20.0, 100.0)
TRANSECT_Y_KM = None
THIN_COLUMN_M = 0.1
PLOT_STRIDE = 1
VERTICAL_EXAGGERATION = 35.0
SAVE_FIGURES = False
FIGURE_DIR = Path("figures") / CASE_NAME

RHO_ICE, RHO_OCEAN = 918.0, 1028.0
SECONDS_PER_DAY = 86400.0
DAYS_PER_YEAR = 365.0
SECONDS_PER_YEAR = DAYS_PER_YEAR * SECONDS_PER_DAY
DT_THERM_SECONDS = 300.0
MELT_DEPTH_THRESHOLD = 10.0
FILL_LARGE = 1.0e30

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "axes.titleweight": "bold"})

def numbered_dirs(root, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)$")
    paths = [p for p in root.iterdir() if p.is_dir() and pattern.match(p.name)] if root.exists() else []
    return sorted(paths, key=lambda p: int(p.name[len(prefix):]))

def nc_array(ds, name, key=slice(None)):
    data = np.asarray(np.ma.filled(ds.variables[name][key], np.nan), dtype="f8")
    data[np.abs(data) > FILL_LARGE] = np.nan
    return data

def file_time(path):
    with netCDF4.Dataset(path) as ds:
        return float(nc_array(ds, "Time", -1))

def timed_dirs(root, prefix, filename):
    pairs = [(file_time(p / filename), p) for p in numbered_dirs(root, prefix) if (p / filename).exists()]
    return [p for _, p in sorted(pairs, key=lambda pair: pair[0])]

def interp_field(years, field, year):
    if year <= years[0]: return field[0].copy()
    if year >= years[-1]: return field[-1].copy()
    hi = int(np.searchsorted(years, year)); lo = hi - 1
    weight = (year - years[lo]) / (years[hi] - years[lo])
    return (1.0 - weight) * field[lo] + weight * field[hi]

def weighted_column_mean(field, h):
    valid = np.isfinite(field) & np.isfinite(h) & (h > 1.0e-8)
    denominator = np.sum(np.where(valid, h, 0.0), axis=0)
    numerator = np.sum(np.where(valid, field * h, 0.0), axis=0)
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan), where=denominator > 0)

def symmetric_limit(field, percentile=99.5):
    values = np.asarray(field)[np.isfinite(field)]
    return max(float(np.percentile(np.abs(values), percentile)), 1.0e-12) if values.size else 1.0

def grounding_contour(ax, grounded, x, y, **kwargs):
    if np.nanmin(grounded) < 0.5 < np.nanmax(grounded):
        ax.contour(x, y, grounded, levels=[0.5], **kwargs)

def show_figure(fig, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()

print("control:", CONTROL)
print("archive:", ARCHIVE)

In [ ]:
if not CONTROL.exists() or not ARCHIVE.exists():
    raise FileNotFoundError("Control or archive path is missing")

input_root = CONTROL / "INPUT" / "generated_geometry"
shelf_file = input_root / f"{CASE_NAME}_shelf_mass.nc"
topog_file = input_root / f"{CASE_NAME}_topog.nc"
summary_file = input_root / f"{CASE_NAME}_summary.json"
metadata = json.loads(summary_file.read_text()) if summary_file.exists() else {}

restart_dirs = timed_dirs(ARCHIVE, "restart", "Shelf.res.nc")
output_dirs = timed_dirs(ARCHIVE, "output", "ice.nc")
if not restart_dirs or not output_dirs:
    raise FileNotFoundError("No timed outputNNN/restartNNN directories found")
latest_restart, latest_output = restart_dirs[-1], output_dirs[-1]

exit_codes = {p.name: (p / "exitcode").read_text().strip() for p in output_dirs if (p / "exitcode").exists()}

with netCDF4.Dataset(shelf_file) as ds:
    source_years = nc_array(ds, "Time") / SECONDS_PER_YEAR
    x_m, y_m = nc_array(ds, "x"), nc_array(ds, "y")
    topog_depth = nc_array(ds, "model_topography_depth")
    prescribed_mass = nc_array(ds, "shelf_mass")
    prescribed_thickness = nc_array(ds, "full_ice_thickness")
    ice_fraction = nc_array(ds, "ice_fraction")
    floating_fraction = nc_array(ds, "floating_fraction")
    grounded_fraction = nc_array(ds, "grounded_fraction")
    expected_cavity = nc_array(ds, "expected_cavity_depth")
with netCDF4.Dataset(topog_file) as ds:
    envelope = nc_array(ds, "wettable_envelope") > 0.5
    generated_wet = nc_array(ds, "generated_wet_mask") > 0.5
with netCDF4.Dataset(latest_restart / "Shelf.res.nc") as ds:
    restart_days = float(nc_array(ds, "Time", -1))
    restart_mass = nc_array(ds, "shelf_mass", -1)
    restart_area = nc_array(ds, "shelf_area", -1)
    restart_ice_h = nc_array(ds, "h_shelf", -1)
with netCDF4.Dataset(latest_restart / "MOM.res.nc") as ds:
    restart_layer_h = nc_array(ds, "h", -1)

x_km, y_km = x_m / 1000.0, y_m / 1000.0
cell_area = abs(float(np.median(np.diff(x_m))) * float(np.median(np.diff(y_m))))
extent = [x_km.min(), x_km.max(), y_km.min(), y_km.max()]
restart_year = restart_days / DAYS_PER_YEAR
applied_year = restart_year - DT_THERM_SECONDS / SECONDS_PER_YEAR

current_input_mass = interp_field(source_years, prescribed_mass, applied_year)
current_thickness = interp_field(source_years, prescribed_thickness, applied_year)
current_ice = interp_field(source_years, ice_fraction, applied_year)
current_floating = interp_field(source_years, floating_fraction, applied_year)
current_grounded = interp_field(source_years, grounded_fraction, applied_year)
model_cavity = np.sum(np.where(np.isfinite(restart_layer_h), restart_layer_h, 0.0), axis=0)
load_cavity = np.maximum(topog_depth - restart_mass / RHO_OCEAN, 0.0)
unloaded = generated_wet & (restart_mass <= 1.0e-12)
water_offset = float(np.median(model_cavity[unloaded] - load_cavity[unloaded]))
adjusted_cavity = np.maximum(load_cavity + water_offset, 0.0)
cavity_residual = model_cavity - adjusted_cavity

summary = pd.DataFrame([
    ("model time", restart_year, "years"),
    ("archived annual outputs", len(output_dirs), "count"),
    ("static wet cells", generated_wet.sum(), "cells"),
    ("wettable-envelope cells", envelope.sum(), "cells"),
    ("prescribed grounded cells", (current_grounded > 0.5).sum(), "cells"),
    ("prescribed floating cells", (current_floating > 0.5).sum(), "cells"),
    ("vanishing envelope columns", (envelope & (model_cavity <= THIN_COLUMN_M)).sum(), "cells"),
    ("applied ice mass", np.nansum(restart_mass * restart_area) / 1e12, "Gt"),
    ("model ocean volume", np.nansum(model_cavity[generated_wet]) * cell_area / 1e9, "km3"),
    ("water-level/datum offset", water_offset, "m"),
    ("datum-adjusted cavity residual p95", np.percentile(np.abs(cavity_residual[generated_wet]), 95), "m"),
    ("datum-adjusted cavity residual max", np.max(np.abs(cavity_residual[generated_wet])), "m"),
    ("max applied-load error", np.max(np.abs(restart_mass - current_input_mass)), "kg m-2"),
    ("max mass - 918*h_shelf", np.max(np.abs(restart_mass - RHO_ICE * restart_ice_h)), "kg m-2"),
], columns=["metric", "value", "units"])
display(summary.style.format({"value": "{:,.6g}"}))

checks = pd.DataFrame({
    "check": ["all exit codes zero", "restart finite", "one-step-offset load matches", "rho_ice consistency", "local cavity residual below 1 m"],
    "passed": [
        bool(exit_codes) and all(v == "0" for v in exit_codes.values()),
        np.isfinite(restart_mass).all() and np.isfinite(restart_layer_h).all(),
        np.max(np.abs(restart_mass - current_input_mass)) < 1e-6,
        np.max(np.abs(restart_mass - RHO_ICE * restart_ice_h)) < 1e-6,
        np.max(np.abs(cavity_residual[generated_wet])) < 1.0,
    ],
})
display(checks)
print(f"latest output: {latest_output.name}; latest restart: {latest_restart.name}; applied geometry year: {applied_year:.9f}")

## Prescribed geometry sequence

Ice thickness and load-derived cavity thickness are shown at selected prescribed years. Red contours mark the grounding line. Year 100 is an input reference, not a simulated state in the current archive.

In [ ]:
snapshot_years = sorted({float(np.clip(y, source_years[0], source_years[-1])) for y in (*SNAPSHOT_YEARS, restart_year)})
snapshots = [{
    "year": year,
    "thick": interp_field(source_years, prescribed_thickness, year),
    "ice": interp_field(source_years, ice_fraction, year),
    "grounded": interp_field(source_years, grounded_fraction, year),
    "cavity": interp_field(source_years, expected_cavity, year),
    "mass": interp_field(source_years, prescribed_mass, year),
} for year in snapshot_years]

fig, axes = plt.subplots(2, len(snapshots), figsize=(4.1 * len(snapshots), 7.2), sharex=True, sharey=True, constrained_layout=True)
ice_vmax = max(np.nanpercentile(s["thick"][s["ice"] > 0.5], 99) for s in snapshots)
cavity_vmax = max(np.nanmax(s["cavity"][generated_wet]) for s in snapshots)
for col, snap in enumerate(snapshots):
    im0 = axes[0, col].imshow(np.ma.masked_where(snap["ice"] <= 0.5, snap["thick"]), origin="lower", extent=extent, aspect="auto", cmap="Blues", vmin=0, vmax=ice_vmax)
    im1 = axes[1, col].imshow(np.ma.masked_where(~generated_wet, snap["cavity"]), origin="lower", extent=extent, aspect="auto", cmap="viridis", vmin=0, vmax=cavity_vmax)
    for ax in axes[:, col]:
        grounding_contour(ax, snap["grounded"], x_km, y_km, colors="tab:red", linewidths=1.1)
        ax.set_xlabel("x (km)")
    axes[0, col].set_title(f"year {snap['year']:g}")
axes[0, 0].set_ylabel("y (km)\nice thickness")
axes[1, 0].set_ylabel("y (km)\ncavity thickness")
fig.colorbar(im0, ax=axes[0, :], label="full ice thickness (m)", shrink=.85)
fig.colorbar(im1, ax=axes[1, :], label="load-derived ocean thickness (m)", shrink=.85)
fig.suptitle("Ocean3 prescribed grounding-line retreat and cavity opening")
show_figure(fig, "prescribed_geometry_snapshots")

j = int(np.argmin(np.abs(y_km - (TRANSECT_Y_KM if TRANSECT_Y_KM is not None else y_km.mean()))))
colors = plt.cm.viridis(np.linspace(.05, .95, len(snapshots)))
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), constrained_layout=True)
axes[0].plot(x_km, -topog_depth[j], color="black", lw=2, label="static MOM bed")
for color, snap in zip(colors, snapshots):
    draft = np.minimum(snap["mass"] / RHO_OCEAN, topog_depth)
    base = np.where(snap["ice"] > .5, -draft, np.nan)
    surface = np.where(snap["ice"] > .5, base + snap["thick"], np.nan)
    axes[0].plot(x_km, base[j], color=color, lw=2, label=f"base y{snap['year']:g}")
    axes[0].plot(x_km, surface[j], color=color, lw=1.1, ls="--")
axes[0].axhline(0, color=".5", lw=.8)
axes[0].set(xlabel="x (km)", ylabel="elevation (m)", title=f"Geometry transect at y={y_km[j]:g} km")
axes[0].legend(ncol=2, fontsize=8)

cavity_ymean = np.nanmean(np.where(envelope[None], expected_cavity, np.nan), axis=1)
grounded_ymean = np.nanmean(grounded_fraction, axis=1)
mesh = axes[1].pcolormesh(x_km, source_years, cavity_ymean, shading="auto", cmap="viridis")
for level, style in zip((.1, .5, .9), (":", "-", "--")):
    if grounded_ymean.min() < level < grounded_ymean.max():
        axes[1].contour(x_km, source_years, grounded_ymean, levels=[level], colors="white", linestyles=style, linewidths=1)
axes[1].axhline(restart_year, color="tab:red", lw=1.5, label=f"latest year {restart_year:g}")
axes[1].set(xlabel="x (km)", ylabel="prescribed year", title="Across-y mean cavity; white = grounded fractions")
axes[1].legend()
fig.colorbar(mesh, ax=axes[1], label="mean cavity thickness (m)")
show_figure(fig, "geometry_transect_hovmoller")

## Annual model-versus-input evolution

NetCDF time is authoritative. Each end-of-year model load is compared with the input one 300 s thermodynamic timestep earlier. Cavity error is evaluated after estimating the model-wide free-surface/datum shift from unloaded wet cells.

In [ ]:
rows = []
for out in output_dirs:
    with netCDF4.Dataset(out / "ice.nc") as ds:
        day = float(nc_array(ds, "Time", -1))
        mass = nc_array(ds, "shelf_mass", -1)
        area = nc_array(ds, "area_shelf_h", -1)
        melt = nc_array(ds, "melt_rate", -1)
        mass_flux = nc_array(ds, "mass_flux", -1)
    with netCDF4.Dataset(out / "prog.nc") as ds:
        h = nc_array(ds, "h", -1)
        temp = nc_array(ds, "temp", -1)
        salt = nc_array(ds, "salt", -1)

    year = day / DAYS_PER_YEAR
    load_year = year - DT_THERM_SECONDS / SECONDS_PER_YEAR
    target_mass = interp_field(source_years, prescribed_mass, load_year)
    target_ground = interp_field(source_years, grounded_fraction, load_year)
    target_float = interp_field(source_years, floating_fraction, load_year)
    H = np.sum(np.where(np.isfinite(h), h, 0.0), axis=0)
    expected = np.maximum(topog_depth - mass / RHO_OCEAN, 0.0)
    open_water = generated_wet & (mass <= 1e-12)
    offset = float(np.median(H[open_water] - expected[open_water]))
    adjusted = np.maximum(expected + offset, 0.0)
    residual = H - adjusted
    melt_mask = np.isfinite(melt) & (mass > 0) & (H >= MELT_DEPTH_THRESHOLD)
    valid_t = np.isfinite(temp) & np.isfinite(h) & (h > 1e-8)
    valid_s = np.isfinite(salt) & np.isfinite(h) & (h > 1e-8)

    rows.append({
        "output": out.name, "year": year,
        "model_mass_gt": np.nansum(mass * area) / 1e12,
        "input_mass_gt": np.nansum(target_mass) * cell_area / 1e12,
        "load_error": np.nanmax(np.abs(mass - target_mass)),
        "grounded": np.count_nonzero(target_ground > .5),
        "floating": np.count_nonzero(target_float > .5),
        "model_volume_km3": np.nansum(H[generated_wet]) * cell_area / 1e9,
        "load_volume_km3": np.nansum(expected[generated_wet]) * cell_area / 1e9,
        "adjusted_volume_km3": np.nansum(adjusted[generated_wet]) * cell_area / 1e9,
        "water_offset_m": offset,
        "residual_p95_m": np.percentile(np.abs(residual[generated_wet]), 95),
        "model_vanished": np.count_nonzero(envelope & (H <= THIN_COLUMN_M)),
        "expected_vanished": np.count_nonzero(envelope & (adjusted <= THIN_COLUMN_M)),
        "mean_melt_m_yr": np.nanmean(melt[melt_mask]),
        "net_melt_gt_yr": np.nansum(mass_flux) * SECONDS_PER_YEAR / 1e12,
        "mean_temp_c": np.sum(np.where(valid_t, temp * h, 0)) / np.sum(np.where(valid_t, h, 0)),
        "mean_salt_psu": np.sum(np.where(valid_s, salt * h, 0)) / np.sum(np.where(valid_s, h, 0)),
    })
annual = pd.DataFrame(rows).sort_values(["year", "output"]).drop_duplicates("year", keep="last").reset_index(drop=True)
display(annual.tail().style.format(precision=5))

fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
axes[0,0].plot(annual.year, annual.input_mass_gt, lw=2, label="input"); axes[0,0].plot(annual.year, annual.model_mass_gt, "o", ms=4, label="MOM6")
axes[0,0].set(xlabel="year", ylabel="ice mass (Gt)", title="Prescribed load"); axes[0,0].legend()
axes[0,1].plot(annual.year, annual.grounded, "o-", label="grounded"); axes[0,1].plot(annual.year, annual.floating, "o-", label="floating")
axes[0,1].set(xlabel="year", ylabel="full cells", title="Ice regime"); axes[0,1].legend()
axes[0,2].plot(annual.year, annual.load_volume_km3, label="load only"); axes[0,2].plot(annual.year, annual.adjusted_volume_km3, label="load + datum"); axes[0,2].plot(annual.year, annual.model_volume_km3, "o", ms=4, label="MOM6")
axes[0,2].set(xlabel="year", ylabel="ocean volume (km3)", title="Static-mask ocean volume"); axes[0,2].legend(fontsize=8)
axes[1,0].plot(annual.year, annual.water_offset_m, "o-", color="tab:blue"); axes[1,0].set(xlabel="year", ylabel="datum offset (m)", title="Water level and local residual")
twin = axes[1,0].twinx(); twin.plot(annual.year, annual.residual_p95_m, "s--", color="tab:red"); twin.set_ylabel("residual p95 (m)", color="tab:red"); twin.grid(False)
axes[1,1].plot(annual.year, annual.model_vanished, "o-", label="MOM6"); axes[1,1].plot(annual.year, annual.expected_vanished, "s--", label="load + datum")
axes[1,1].set(xlabel="year", ylabel=f"envelope cells H <= {THIN_COLUMN_M:g} m", title="Vanishing columns"); axes[1,1].legend()
axes[1,2].plot(annual.year, annual.net_melt_gt_yr, "o-", color="tab:purple"); axes[1,2].set(xlabel="year", ylabel="net basal flux (Gt/yr)", title="Ice-ocean thermodynamics")
twin = axes[1,2].twinx(); twin.plot(annual.year, annual.mean_melt_m_yr, "s--", color="tab:orange"); twin.set_ylabel("mean active melt (m/yr)", color="tab:orange"); twin.grid(False)
show_figure(fig, "annual_geometry_melt")

In [ ]:
frames = []
for out in output_dirs:
    path = out / "ocean.stats.nc"
    if path.exists():
        with netCDF4.Dataset(path) as ds:
            frames.append(pd.DataFrame({
                "day": nc_array(ds, "Time"), "ke": np.sum(nc_array(ds, "KE"), axis=1),
                "mass": nc_array(ds, "Mass"), "salt": nc_array(ds, "Salt"), "heat": nc_array(ds, "Heat"),
                "cfl_t": nc_array(ds, "max_CFL_trans"), "cfl_l": nc_array(ds, "max_CFL_lin"),
                "ntrunc": nc_array(ds, "Ntrunc"), "output": out.name,
            }))
stats = pd.concat(frames).sort_values(["day", "output"]).drop_duplicates("day", keep="last").reset_index(drop=True)
stats["year"] = stats.day / DAYS_PER_YEAR
stats["ke_tj"] = stats.ke / 1e12
stats["mass_d_gt"] = (stats.mass - stats.mass.iloc[0]) / 1e12
stats["salt_d_gt"] = (stats.salt - stats.salt.iloc[0]) / 1e12
stats["heat_d_ej"] = (stats.heat - stats.heat.iloc[0]) / 1e18

fig, axes = plt.subplots(2,2,figsize=(15,8),constrained_layout=True)
axes[0,0].plot(stats.year, stats.ke_tj); axes[0,0].set(xlabel="year",ylabel="total KE (TJ)",title="Kinetic energy")
axes[0,1].plot(stats.year, stats.mass_d_gt); axes[0,1].set(xlabel="year",ylabel="change (Gt)",title="Ocean mass")
axes[1,0].plot(stats.year, stats.salt_d_gt,color="tab:blue"); axes[1,0].set(xlabel="year",ylabel="salt change (Gt)",title="Salt and heat content")
twin=axes[1,0].twinx(); twin.plot(stats.year,stats.heat_d_ej,color="tab:red"); twin.set_ylabel("heat change (EJ)",color="tab:red"); twin.grid(False)
axes[1,1].plot(stats.year,stats.cfl_t,label="transport"); axes[1,1].plot(stats.year,stats.cfl_l,ls="--",label="linear")
axes[1,1].set(xlabel="year",ylabel="maximum CFL",title=f"Stability; max Ntrunc={stats.ntrunc.max():g}"); axes[1,1].legend()
show_figure(fig,"ocean_stats")
display(pd.DataFrame({"metric":["max transport CFL","max linear CFL","max Ntrunc","final KE","final ocean mass change"],"value":[stats.cfl_t.max(),stats.cfl_l.max(),stats.ntrunc.max(),stats.ke_tj.iloc[-1],stats.mass_d_gt.iloc[-1]],"units":["1","1","count","TJ","Gt"]}).style.format({"value":"{:,.6g}"}))

## Latest ocean state

Maps use the final five-day-mean ocean record, final daily pressure, and final annual ice record. Face velocities are averaged to tracer cells and summarized as a thickness-weighted column RMS.

In [ ]:
with netCDF4.Dataset(latest_output / "ave_prog.nc") as ds:
    mean_day = float(nc_array(ds,"Time",-1))
    mean_h, mean_e = nc_array(ds,"h",-1), nc_array(ds,"e",-1)
    mean_temp, mean_salt = nc_array(ds,"temp",-1), nc_array(ds,"salt",-1)
    mean_u, mean_v = nc_array(ds,"u",-1), nc_array(ds,"v",-1)
with netCDF4.Dataset(latest_output / "forcing.nc") as ds:
    pressure_day = float(nc_array(ds,"Time",-1)); pressure = nc_array(ds,"p_surf",-1)
with netCDF4.Dataset(latest_output / "ice.nc") as ds:
    latest_melt = nc_array(ds,"melt_rate",-1)

uc = .5*(mean_u[:,:,:-1]+mean_u[:,:,1:])
vc = .5*(mean_v[:,:-1,:]+mean_v[:,1:,:])
speed = np.sqrt(uc**2+vc**2)
mean_H = np.sum(np.where(np.isfinite(mean_h),mean_h,0),axis=0)
speed_rms = np.sqrt(weighted_column_mean(speed**2,mean_h))
temp_bar, salt_bar = weighted_column_mean(mean_temp,mean_h), weighted_column_mean(mean_salt,mean_h)
wet_now = mean_H > THIN_COLUMN_M
melt_now = (restart_mass > 0) & (model_cavity >= MELT_DEPTH_THRESHOLD)

fields = [
    (np.ma.masked_where(~wet_now,mean_H),"ocean column","m","viridis",False),
    (np.ma.masked_where(~wet_now,100*speed_rms),"column-RMS speed","cm/s","magma",False),
    (np.ma.masked_where(~wet_now,temp_bar),"mean temperature","degC","coolwarm",False),
    (np.ma.masked_where(~wet_now,salt_bar),"mean salinity","psu","viridis",False),
    (np.ma.masked_where(~melt_now,latest_melt),"basal melt rate","m/yr","coolwarm",True),
    (np.ma.masked_where(~generated_wet,pressure/1e6),"surface/load pressure","MPa","cividis",False),
]
fig,axes=plt.subplots(2,3,figsize=(17,8.5),constrained_layout=True)
for ax,(field,title,units,cmap,symmetric) in zip(axes.ravel(),fields):
    kwargs={}
    if symmetric:
        limit=symmetric_limit(field); kwargs={"vmin":-limit,"vmax":limit}
    im=ax.imshow(field,origin="lower",extent=extent,aspect="auto",cmap=cmap,**kwargs)
    grounding_contour(ax,current_grounded,x_km,y_km,colors="white",linewidths=.8)
    ax.set(title=title,xlabel="x (km)",ylabel="y (km)"); fig.colorbar(im,ax=ax,label=units)
fig.suptitle(f"Latest ocean mean: year {mean_day/365:.3f}; pressure: year {pressure_day/365:.3f}")
show_figure(fig,"latest_ocean_state")

section_h=mean_h[:,j,:]
section_z=.5*(mean_e[:-1,j,:]+mean_e[1:,j,:])
section_speed=100*speed[:,j,:]
section_temp=mean_temp[:,j,:]
x2=np.broadcast_to(x_km,section_z.shape)
valid=np.isfinite(section_z)&np.isfinite(section_h)&(section_h>.05)
current_base=np.where(restart_mass>0,-np.minimum(restart_mass/RHO_OCEAN,topog_depth),np.nan)
fig,axes=plt.subplots(2,1,figsize=(15,9),sharex=True,sharey=True,constrained_layout=True)
for ax,values,cmap,label in [(axes[0],section_speed,"magma","speed (cm/s)"),(axes[1],section_temp,"coolwarm","potential temperature (degC)")]:
    lo,hi=np.nanpercentile(values[valid],[1,99]); lo=0 if label.startswith("speed") else lo
    levels=np.linspace(lo,max(hi,lo+1e-6),31)
    cf=ax.contourf(x2,section_z,np.ma.masked_where(~valid,values),levels=levels,cmap=cmap,extend="max" if label.startswith("speed") else "both")
    for k in range(0,mean_e.shape[0],6): ax.plot(x_km,mean_e[k,j],color="white",lw=.35,alpha=.5)
    ax.plot(x_km,-topog_depth[j],color="black",lw=1.5,label="bed"); ax.plot(x_km,current_base[j],color="cyan",lw=1.3,label="ice base")
    ax.set(ylabel="elevation (m)",title=label); fig.colorbar(cf,ax=ax,label=label)
axes[0].legend(loc="lower right"); axes[1].set_xlabel("x (km)")
fig.suptitle(f"Latest five-day-mean section at y={y_km[j]:g} km")
show_figure(fig,"latest_ocean_section")

In [ ]:
def plot_3d_current():
    if go is None:
        print("plotly unavailable")
        return None
    stride=max(int(PLOT_STRIDE),1)
    xx,yy=np.meshgrid(x_km[::stride],y_km[::stride])
    bed=np.where(generated_wet,-topog_depth,np.nan)[::stride,::stride]
    ice=restart_mass>0
    base=np.where(ice,-np.minimum(restart_mass/RHO_OCEAN,topog_depth),np.nan)[::stride,::stride]
    surface=np.where(ice,base+restart_ice_h,np.nan)[::stride,::stride]
    fig=go.Figure()
    fig.add_surface(x=xx,y=yy,z=bed,name="model bed",colorscale="Greys",opacity=.65,showscale=False)
    fig.add_surface(x=xx,y=yy,z=base,name="ice base",colorscale="Blues",opacity=.75,showscale=False)
    fig.add_surface(x=xx,y=yy,z=surface,name="ice surface",colorscale="Ice",opacity=.45,showscale=False)
    xr,yr,zr=float(np.ptp(x_km)),float(np.ptp(y_km)),float(np.nanmax(surface)-np.nanmin(bed))
    fig.update_layout(title=f"Load-derived geometry at model year {restart_year:g}",height=720,margin=dict(l=0,r=0,b=0,t=55),
        scene=dict(xaxis_title="x (km)",yaxis_title="y (km)",zaxis_title="elevation (m)",aspectmode="manual",
        aspectratio=dict(x=1,y=max(yr/xr,.05),z=max(VERTICAL_EXAGGERATION*zr/(1000*xr),.08))))
    return fig
geometry_3d=plot_3d_current()
if geometry_3d is not None:
    geometry_3d.show()

## Interpretation notes

- Shelf mass is compared at one 300 s thermodynamic step before each timestamp, matching MOM6 prescribed-movement semantics.
- Grounded/floating counts come from input masks. With READ_SHELF_AREA=False, area_shelf_h covers the fixed 6,400-cell ice footprint and is not floating area.
- Columns at or below 0.1 m are treated as effectively vanished; sub-centimetre marginal thickness can remain by design.
- Because CONST_SEA_LEVEL=False, opening cavity volume lowers the model-wide water level. Local error must be assessed against max(topography minus load plus diagnosed datum offset, 0).
- This production experiment includes stratification, shelf thermodynamics, a tracer sponge, Coriolis, and evolving load. Use the matched resting controls before attributing velocity to pressure-gradient truncation.
- The current simulated year is read from Shelf.res.nc. Year 100 is only the prescribed endpoint reference.